# Word2vec
## workNet
使用计算机处理文本词汇，一种处理方式是**WordNet**，构建一个包含`同义词集`和`上位词("is a"关系)`的列表的词典

In [6]:
from nltk.corpus import wordnet as wn
poses = { 'n':'noun', 'v':'verb', 's':'adj (s)', 'a':'adj', 'r':'adv'}
for synset in wn.synsets("good"):
    print("{}: {}".format(poses[synset.pos()], ", ".join([l.name() for l in synset.lemmas()])))

noun: good
noun: good, goodness
noun: good, goodness
noun: commodity, trade_good, good
adj: good
adj (s): full, good
adj: good
adj (s): estimable, good, honorable, respectable
adj (s): beneficial, good
adj (s): good
adj (s): good, just, upright
adj (s): adept, expert, good, practiced, proficient, skillful, skilful
adj (s): good
adj (s): dear, good, near
adj (s): dependable, good, safe, secure
adj (s): good, right, ripe
adj (s): good, well
adj (s): effective, good, in_effect, in_force
adj (s): good
adj (s): good, serious
adj (s): good, sound
adj (s): good, salutary
adj (s): good, honest
adj (s): good, undecomposed, unspoiled, unspoilt
adj (s): good
adv: well, good
adv: thoroughly, soundly, good


In [8]:
panda = wn.synset("panda.n.01")
hyper = lambda s: s.hypernyms()
list(panda.closure(hyper))

[Synset('procyonid.n.01'),
 Synset('carnivore.n.01'),
 Synset('placental.n.01'),
 Synset('mammal.n.01'),
 Synset('vertebrate.n.01'),
 Synset('chordate.n.01'),
 Synset('animal.n.01'),
 Synset('organism.n.01'),
 Synset('living_thing.n.01'),
 Synset('whole.n.02'),
 Synset('object.n.01'),
 Synset('physical_entity.n.01'),
 Synset('entity.n.01')]

## 单词的离散表征
#### WordNet 有如下问题
1. 忽略了词汇的细微差别
2. 缺少单词的新含义
3. 是人为构建的，有一定的主观性，需要付出足够的人力成本
4. 无法定量计算单词的相似程度
#### one-hot vectors：一种离散表示形式

In [9]:
motel = [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]
hotel = [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]

## Word2vec
### one-hot vectors的问题
所有vectors正交，无法评估词之间的相似性
### word2vec overview
- 从一个较大的语料库(large `corpus`)开始（a long list of words）
- 所有词汇用一个 vector 表征，每个 vector 有一个随机的初始值（**注意：不能全是0**）
- 遍历`corpus`中所有位置 $t$，每个位置有核心词 $c$ 和上下文(context / outside) $o$
- 利用 $c$ 与 $o$ 词汇的相似性，**给定 $c$ (或 $o$), 计算 $o$ (或 $c$) 的概率**
- 不停调整 word vectors，让这个概率最大化

### 量化这个过程
$P(w_{t+j}|w_{t})$: 位置 $t$ 出现词汇 $w_{t}$的前提下，位置 $t+j$ 出现词汇 $w_{t+j}$ 的概率  
对每一个位置 $t = 1, ..., T$，在一个固定大小( $m$ )的上下文(context)窗口中，对每一个中心词 $w_t$：
$$
Likelihood = L(\theta) = \prod_{t=1}^T \prod_{\substack{-m \le j \le m \\ j \ne 0}}P(w_{t+j}|w_t;\theta)
$$
基于上面这个概率，引入**Objective function** $J(\theta)$：
$$
J(\theta) = -\frac{1}{T}logL(\theta) = -\frac{1}{T}\sum_{t=1}^T \sum_{\substack{-m \le j \le m \\ j \ne 0}}logP(w_{t+j}|w_t;\theta)
$$
- $J(\theta)$ 是 $L(\theta)$ 的负对数平均，对数是为了方便求导(Gradient Descent)，负数是为了从 Maximize 转换成 Minimize（更符合习惯）

#### $P(w_{t+j}|w_{t})$ 的计算
- $v_w$: 表示 $w$ 是 center word
- $u_w$: 表示 $w$ 是上下文(context word)
$$
P(o|c) = \frac{exp(u_o^Tv_c)}{\sum_{w\in V}exp(u_w^Tv_c)}
$$
也就是使用 softmax function $\mathbb{R} ^n\to(0, 1)^n$

### Optimization：Gradient Descent
- $\theta$ 表示了所有`model parameters`，$\theta = [v_1, v_2, ...v_n, u_1, u_2, ...u_n]^T \in \mathbb{R}^{2dV}$
- 每个 word 都有两个 vectors
  - $u_i$: when $w_i$ is outside/context word
  - $v_i$: when $w_i$ is center word
Update epuation:
$$
\theta^{new} = \theta^{old} - \alpha\nabla_{\theta}J(\theta)
$$
> 如果一个 corpus 中有 $d$ 个 word，每个 word 有 $V$ 维度，那么一共会有 $2dV$ 个参数  
> 需要对这些 parameters 同时进行更新
#### Stochastic gradient descent(SGD)
> 解决数据过多的问题，每次取一个 "Batch" ，计算这个子集的梯度当作整体数据集的梯度
```python
while True:
    window = sample_window(corpus)
    theta_grad = evaluate_gradient(J, window, theta)
    theta = theta - alpha * theta_grad
```